# Fake News Prediction using Machine Learning

In this notebook, we build an end-to-end Machine Learning pipeline to detect **Fake News** using **Natural Language Processing (Stemming & TF-IDF Vectorization)** and **Logistic Regression**.


### Importing the Dependencies


In [ ]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix


### Data Collection and Preprocessing


In [ ]:
# Sample dataset representing news articles
news_samples = {
    'id': range(1, 13),
    'title': [
        'House Republicans Fret Over Bill Clinton Speaking at Convention',
        'Flynn Secretly Promised to Kick Russian Diplomats Out',
        'FBI Releases Declassified Documents on Trump Tower Wiretaps',
        'CERN Scientists Discover Gateway to Parallel Universe',
        'Pope Francis Endorses Donald Trump for US Presidency',
        'NASA Confirms Earth Will Experience 15 Days of Total Darkness',
        'Federal Reserve Announces Modest Interest Rate Increase',
        'Secret Government Lab Releases Genetically Modified Mosquitoes',
        'Supreme Court Reaches Landmark Decision on Voting Rights',
        'Celebrity Chef Reveals Secret Recipe That Cures All Diseases',
        'Stock Markets Rally Following Strong Employment Data',
        'Drinking Lemon Water Every Morning Eliminates 100% of Body Toxins'
    ],
    'author': [
        'Darrell Lucus', 'Daniel Greenfield', 'Consortiumnews.com', 'Alex Jones',
        'American News', 'World News Daily', 'Reuters Business', 'Infowars',
        'Associated Press', 'Viral Health Online', 'Bloomberg News', 'Health Buzz'
    ],
    'label': [0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1]  # 1: Fake, 0: Real
}

news_dataset = pd.DataFrame(news_samples)
news_dataset.head()


In [ ]:
# Shape of dataset
print("Dataset Shape:", news_dataset.shape)
news_dataset['label'].value_counts()


### 1 --> Fake News
### 0 --> Real News


### Data Visualization


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='label', data=news_dataset, palette=['#2ecc71', '#e74c3c'])
plt.title('Distribution of Real (0) vs Fake (1) News')
plt.xticks(ticks=[0, 1], labels=['Real News (0)', 'Fake News (1)'])
plt.show()


### Merging Author & Title as Content Feature


In [ ]:
news_dataset['content'] = news_dataset['author'] + ' ' + news_dataset['title']
print(news_dataset['content'][0])


### Stemming: Reducing Words to their Root Words


In [ ]:
port_stem = PorterStemmer()

def stemming(content):
    # Remove non-alphabet characters
    stemmed_content = re.sub('[^a-zA-Z]', ' ', content)
    stemmed_content = stemmed_content.lower().split()
    # Simple English stopwords list
    stop_words = {'the', 'a', 'an', 'in', 'on', 'at', 'to', 'for', 'of', 'and', 'or', 'is', 'are', 'was'}
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if word not in stop_words]
    return ' '.join(stemmed_content)

news_dataset['content'] = news_dataset['content'].apply(stemming)
print("Stemmed content example:", news_dataset['content'][0])


### Separating Features and Target


In [ ]:
X = news_dataset['content'].values
Y = news_dataset['label'].values
print(X)


### Converting Text to Numerical Data (TF-IDF Vectorization)


In [ ]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(X)
print("Feature Matrix Shape:", X.shape)


### Splitting into Training and Test Data


In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=2)
print("Train:", X_train.shape, "Test:", X_test.shape)


### Model Training --> Logistic Regression


In [ ]:
model = LogisticRegression()
model.fit(X_train, Y_train)


### Model Evaluation


In [ ]:
# Accuracy on training data
X_train_pred = model.predict(X_train)
print('Accuracy on Training data : ', round(accuracy_score(Y_train, X_train_pred) * 100, 2), '%')

# Accuracy on test data
X_test_pred = model.predict(X_test)
print('Accuracy on Test data     : ', round(accuracy_score(Y_test, X_test_pred) * 100, 2), '%')


### Building a Predictive System


In [ ]:
sample_headline = ["Darrell Lucus Federal Reserve Confirms Modest Interest Rate Adjustment for Economy"]

# Preprocess and vectorize
stemmed_sample = [stemming(sample_headline[0])]
sample_features = vectorizer.transform(stemmed_sample)

prediction = model.predict(sample_features)
print("Prediction output:", prediction)

if prediction[0] == 0:
    print('>>> Result: The news is REAL (Verified Information).')
else:
    print('>>> Result: The news is FAKE (Fabricated / Clickbait).')
